# PicoCal — Retrain: kNN-9 vs kNN-25, energy cut 1-100 GeV (notebook 08)

Mentors chose 5x5 nearest (kNN-25); nb 07 found kNN-9 best on clean signal. Retrain both with the new energy cut `1 <= E_true <= 100 GeV`, train all regions / test R3.

## A. kNN-9 vs kNN-25 retrain

In [9]:
import sys, copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.ensemble import HistGradientBoostingRegressor

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import build, select_knn, split, resolution, TokenDS, collate, EPS

FILES = 100
EPOCHS = 250
SEEDS = 5
EMIN, EMAX = 1.0, 100.0
cfg = {"d": 96, "nhead": 4, "layers": 3, "dropout": 0.1, "lr": 3e-4, "wd": 1e-4,
       "batch": 128, "epochs": EPOCHS, "patience": 60}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
files = sorted((repo / "data" / "full").glob("matched_*.root"))[:FILES]
{"device": DEVICE, "files": len(files), "energy_cut_GeV": [EMIN, EMAX], "seeds": SEEDS}

{'device': 'cuda', 'files': 100, 'energy_cut_GeV': [1.0, 100.0], 'seeds': 5}

In [10]:
class TunedTransformer(nn.Module):
    def __init__(self, in_dim, d=96, nhead=4, layers=3, dropout=0.1):
        super().__init__()
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, nhead, dim_feedforward=4 * d, dropout=dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d, 1))

    def forward(self, x, m):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = m.unsqueeze(-1).float()
        return self.head((h * w).sum(1) / w.sum(1).clamp(min=1))

In [11]:
def train_one(make_model, train_idx, toks, y, rva, rte, Et, seed, verbose=False):
    torch.manual_seed(seed)
    cont = np.concatenate([toks[i][:, :7] for i in train_idx], 0)
    mean = cont.mean(0); std = cont.std(0) + EPS

    def loader(idx, sh):
        return DataLoader(TokenDS([toks[i] for i in idx], y[idx], mean, std, 7),
                          batch_size=cfg["batch"], shuffle=sh, collate_fn=collate)

    model = make_model().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    dl_tr, dl_va = loader(train_idx, True), loader(rva, False)

    def vloss():
        model.eval(); t = 0.0; n = 0
        with torch.no_grad():
            for X, m, yb in dl_va:
                t += nn.functional.mse_loss(model(X.to(DEVICE), m.to(DEVICE)), yb.to(DEVICE)).item(); n += 1
        return t / max(n, 1)

    hist = {"epoch": [], "train_loss": [], "val_loss": []}
    best = float("inf"); best_state = None; wait = 0
    for ep in range(cfg["epochs"]):
        model.train(); tl = 0.0; nb = 0
        for X, m, yb in dl_tr:
            opt.zero_grad()
            loss = nn.functional.mse_loss(model(X.to(DEVICE), m.to(DEVICE)), yb.to(DEVICE))
            loss.backward(); opt.step()
            tl += loss.item(); nb += 1
        sched.step()
        v = vloss(); tr = tl / max(nb, 1)
        hist["epoch"].append(ep); hist["train_loss"].append(tr); hist["val_loss"].append(v)
        if verbose:
            print(f"  epoch {ep:3d}  train_loss {tr:.4f}  val_loss {v:.4f}", flush=True)
        if v < best - 1e-4:
            best = v; best_state = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= cfg["patience"]:
                break
    model.load_state_dict(best_state)

    def predict(idx):
        model.eval(); out = []
        with torch.no_grad():
            for X, m, _ in loader(idx, False):
                out.append(model(X.to(DEVICE), m.to(DEVICE)).cpu().numpy().ravel())
        return np.concatenate(out)

    pv, pt = predict(rva), predict(rte)
    a, b = np.polyfit(pv, y[rva], 1)
    model._mean = mean; model._std = std
    return resolution(np.exp(a * pt + b), Et[rte])["sigma_eff"], hist, model

In [12]:
def build_sel(k):
    D = build(files, 3, 100.0, selector=lambda c: select_knn(c, k))
    keep = (D["Etrue"] >= EMIN) & (D["Etrue"] <= EMAX)
    return D, keep

def evaluate(D, keep, k):
    y = D["y"]; Et = D["Etrue"]; toks = D["tok_seed"]
    ridx = np.flatnonzero((D["region"] == 3) & keep)
    rtr, rva, rte = (ridx[s] for s in split(len(ridx)))
    ttr = np.setdiff1d(np.setdiff1d(np.flatnonzero(keep), rte), rva)
    in_dim = toks[int(ridx[0])].shape[1]
    mk = lambda: TunedTransformer(in_dim, cfg["d"], cfg["nhead"], cfg["layers"], cfg["dropout"])
    vals = []; hist0 = None; model0 = None
    for s in range(SEEDS):
        print(f"k={k} seed {s}", flush=True)
        sig, h, mdl = train_one(mk, ttr, toks, y, rva, rte, Et, s, verbose=(s == 0))
        vals.append(sig)
        if s == 0:
            hist0 = h; model0 = mdl
    gb = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(D["agg"][rtr], y[rtr])
    bdt = resolution(np.exp(gb.predict(D["agg"][rte])), Et[rte])["sigma_eff"]
    a, b = np.polyfit(np.log(D["total_energy"][rtr] + EPS), y[rtr], 1)
    te = resolution(np.exp(a * np.log(D["total_energy"][rte] + EPS) + b), Et[rte])["sigma_eff"]
    return {"k": k, "n_train_all": int(len(ttr)), "n_test_R3": int(len(rte)),
            "transformer_mean": round(float(np.mean(vals)), 4), "transformer_std": round(float(np.std(vals)), 4),
            "BDT": round(float(bdt), 4), "total_energy": round(float(te), 4),
            "vals": [round(float(v), 4) for v in vals], "history": hist0,
            "model": model0, "rte": rte, "rva": rva, "toks": toks, "Et": Et, "y": y}

res = {}
for k in [9, 25]:
    D, keep = build_sel(k)
    res[k] = evaluate(D, keep, k)
    print(k, res[k], flush=True)
res

k=9 seed 0
  epoch   0  train_loss 0.4904  val_loss 0.1149
  epoch   1  train_loss 0.1316  val_loss 0.0822
  epoch   2  train_loss 0.1220  val_loss 0.0799
  epoch   3  train_loss 0.1153  val_loss 0.0764
  epoch   4  train_loss 0.1099  val_loss 0.0722
  epoch   5  train_loss 0.1074  val_loss 0.0905
  epoch   6  train_loss 0.1058  val_loss 0.0720
  epoch   7  train_loss 0.1035  val_loss 0.0707
  epoch   8  train_loss 0.1009  val_loss 0.0702
  epoch   9  train_loss 0.1031  val_loss 0.0876
  epoch  10  train_loss 0.1011  val_loss 0.0855
  epoch  11  train_loss 0.1019  val_loss 0.0766
  epoch  12  train_loss 0.0994  val_loss 0.0701
  epoch  13  train_loss 0.0995  val_loss 0.0686
  epoch  14  train_loss 0.1012  val_loss 0.0702
  epoch  15  train_loss 0.0995  val_loss 0.0707
  epoch  16  train_loss 0.0972  val_loss 0.0739
  epoch  17  train_loss 0.1000  val_loss 0.0692
  epoch  18  train_loss 0.0981  val_loss 0.0806
  epoch  19  train_loss 0.0973  val_loss 0.0797
  epoch  20  train_loss 0.096

{9: {'k': 9,
  'n_train_all': 30102,
  'n_test_R3': 1686,
  'transformer_mean': 0.0472,
  'transformer_std': 0.0014,
  'BDT': 0.0557,
  'total_energy': 0.0574,
  'vals': [0.0448, 0.0477, 0.0465, 0.0486, 0.0485],
  'history': {'epoch': [0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    25,
    26,
    27,
    28,
    29,
    30,
    31,
    32,
    33,
    34,
    35,
    36,
    37,
    38,
    39,
    40,
    41,
    42,
    43,
    44,
    45,
    46,
    47,
    48,
    49,
    50,
    51,
    52,
    53,
    54,
    55,
    56,
    57,
    58,
    59,
    60,
    61,
    62,
    63,
    64,
    65,
    66,
    67,
    68,
    69,
    70,
    71,
    72,
    73,
    74,
    75,
    76,
    77,
    78,
    79,
    80,
    81,
    82,
    83,
    84,
    85,
    86,
    87,
    88,
    89,
    90,
    91,
    92,
    93,
    94,
    95,
    96,
  

In [13]:
import pandas as pd
pd.DataFrame([{"cells": f"kNN-{k}", "n_train": r["n_train_all"], "n_test_R3": r["n_test_R3"],
               "transformer": r["transformer_mean"], "±": r["transformer_std"],
               "BDT": r["BDT"], "total_energy": r["total_energy"]} for k, r in res.items()])

,cells,n_train,n_test_R3,transformer,±,BDT,total_energy
0,kNN-9,30102,1686,0.0472,0.0014,0.0557,0.0574
1,kNN-25,30061,1706,0.0548,0.0037,0.0497,0.0535


### Per-epoch training metrics (seed 0)

In [14]:
import pandas as pd
frames = []
for k in [9, 25]:
    h = res[k]["history"]
    frames.append(pd.DataFrame({"epoch": h["epoch"],
                                f"kNN{k}_train": np.round(h["train_loss"], 4),
                                f"kNN{k}_val": np.round(h["val_loss"], 4)}).set_index("epoch"))
pd.concat(frames, axis=1)

,kNN9_train,kNN9_val,kNN25_train,kNN25_val
epoch,,,,
0,0.4904,0.1149,0.4995,0.0882
1,0.1316,0.0822,0.1357,0.0805
2,0.1220,0.0799,0.1255,0.0791
3,0.1153,0.0764,0.1206,0.1002
4,0.1099,0.0722,0.1162,0.0707
...,...,...,...,...
125,0.0632,0.0732,NaN,NaN
126,0.0635,0.0716,NaN,NaN
127,0.0641,0.0720,NaN,NaN


In [15]:
import plotly.graph_objects as go
fig = go.Figure()
for k, tc, vc in [(9, "#4c78a8", "#e45756"), (25, "#72b7b2", "#f58518")]:
    h = res[k]["history"]
    fig.add_trace(go.Scatter(x=h["epoch"], y=h["train_loss"], mode="lines",
                             name=f"kNN-{k} train", line=dict(color=tc)))
    fig.add_trace(go.Scatter(x=h["epoch"], y=h["val_loss"], mode="lines",
                             name=f"kNN-{k} val", line=dict(color=vc, dash="dash")))
fig.update_layout(template="plotly_white", height=440, legend_title="",
                  title="Learning curves (seed 0): train vs validation loss",
                  xaxis_title="epoch", yaxis_title="MSE loss (log-energy)")
fig.show()

In [16]:
import plotly.graph_objects as go
labels = ["transformer<br>kNN-9", "transformer<br>kNN-25", "BDT<br>kNN-9", "BDT<br>kNN-25", "LHCb total_energy"]
vals = [res[9]["transformer_mean"], res[25]["transformer_mean"], res[9]["BDT"], res[25]["BDT"], res[9]["total_energy"]]
errs = [res[9]["transformer_std"], res[25]["transformer_std"], 0, 0, 0]
colors = ["#2ca02c", "#4c78a8", "#8c8c8c", "#8c8c8c", "#d62728"]
fig = go.Figure(go.Bar(x=labels, y=vals, marker_color=colors,
                       error_y=dict(type="data", array=errs, color="#333"),
                       text=[f"{v:.4f}" for v in vals], textposition="outside"))
fig.update_layout(template="plotly_white", height=460,
                  title="R3 resolution after energy cut (lower is better): kNN-9 vs kNN-25",
                  yaxis_title="sigma_eff")
fig.show()

## D. Attention on the R3 test set (kNN-25)

In [17]:
import plotly.graph_objects as go

K_ATTN = 25
r = res[K_ATTN]
model = r["model"].eval()
toks = r["toks"]; Et = r["Et"]; yv = r["y"]; rte = r["rte"]; rva = r["rva"]
mean, std = model._mean, model._std

def make_batch(idxs):
    raw = [toks[i] for i in idxs]
    tens = []
    for t in raw:
        tn = t.copy(); tn[:, :7] = (tn[:, :7] - mean) / std
        tens.append((torch.tensor(tn), torch.tensor([0.0])))
    X, m, _ = collate(tens)
    return raw, X.to(DEVICE), m.to(DEVICE)

def attn_received(X, m):
    with torch.no_grad():
        h = model.embed(X)
        _, w = model.enc.layers[0].self_attn(h, h, h, key_padding_mask=~m,
                                             need_weights=True, average_attn_weights=True)
    recv = (w * m.unsqueeze(1).float()).sum(1) / m.unsqueeze(1).float().sum(1).clamp(min=1)
    return (recv * m.float()).cpu().numpy()

_, Xv, mv = make_batch(rva)
with torch.no_grad():
    pv = model(Xv, mv).cpu().numpy().ravel()
a_cal, b_cal = np.polyfit(pv, yv[rva], 1)

raw_te, Xte, mte = make_batch(rte)
recv = attn_received(Xte, mte)
with torch.no_grad():
    pt = model(Xte, mte).cpu().numpy().ravel()
Epred = np.exp(a_cal * pt + b_cal)

dists, attns = [], []
for bi, t in enumerate(raw_te):
    L = t.shape[0]
    dists.extend(t[:, 5].tolist()); attns.extend(recv[bi, :L].tolist())
dists = np.array(dists); attns = np.array(attns)
edges = np.linspace(0, 3, 13); ctr = (edges[:-1] + edges[1:]) / 2
binmean = [float(attns[(dists >= edges[i]) & (dists < edges[i + 1])].mean())
           if ((dists >= edges[i]) & (dists < edges[i + 1])).any() else None for i in range(len(ctr))]
fig = go.Figure()
fig.add_trace(go.Scatter(x=dists, y=attns, mode="markers",
                         marker=dict(size=3, opacity=0.12, color="#4c78a8"), name="cells"))
fig.add_trace(go.Scatter(x=ctr, y=binmean, mode="lines+markers",
                         line=dict(color="#e45756", width=3), name="binned mean"))
fig.update_layout(template="plotly_white", height=430,
                  title="Attention received vs distance-to-seed (kNN-25, R3 test)",
                  xaxis_title="distance / seed pitch", yaxis_title="attention received")
fig.show()

In [18]:
from plotly.subplots import make_subplots

picks = list(range(6))
titles = [f"pred {Epred[i]:.1f} / true {Et[rte[i]]:.1f} GeV" for i in picks]
fig = make_subplots(rows=2, cols=3, subplot_titles=titles, horizontal_spacing=0.07, vertical_spacing=0.13)
for n, i in enumerate(picks):
    t = raw_te[i]; L = t.shape[0]
    rx = t[:, 3]; ry = t[:, 4]; e = np.expm1(t[:, 0]); aw = recv[i, :L]
    fig.add_trace(go.Scatter(x=rx, y=ry, mode="markers",
                             marker=dict(size=8 + 20 * e / (e.max() + 1e-9), color=aw,
                                         colorscale="Viridis", showscale=(n == 0),
                                         colorbar=dict(title="attn", len=0.45, y=0.79)),
                             showlegend=False), row=n // 3 + 1, col=n % 3 + 1)
fig.update_layout(template="plotly_white", height=560,
                  title="R3 test clusters (kNN-25): position seed-pitch units, size=energy, color=attention")
fig.update_xaxes(title_text="rel_x/pitch"); fig.update_yaxes(title_text="rel_y/pitch")
fig.show()

In [20]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

K = 9
rr = res[K]
model = rr["model"].eval()
toks = rr["toks"]; Et = rr["Et"]; yv = rr["y"]; rte = rr["rte"]; rva = rr["rva"]
mean, std = model._mean, model._std

def predict(idxs):
    tens = []
    for i in idxs:
        t = toks[i].copy(); t[:, :7] = (t[:, :7] - mean) / std
        tens.append((torch.tensor(t), torch.tensor([0.0])))
    X, m, _ = collate(tens)
    with torch.no_grad():
        return model(X.to(DEVICE), m.to(DEVICE)).cpu().numpy().ravel()

pv = predict(rva); a, b = np.polyfit(pv, yv[rva], 1)
pt = predict(rte); Epred = np.exp(a * pt + b); Etrue = Et[rte]

fig = make_subplots(rows=1, cols=2, subplot_titles=["predicted vs true energy", "resolution vs true energy"])
fig.add_trace(go.Scatter(x=Etrue, y=Epred, mode="markers",
                         marker=dict(size=4, opacity=0.3, color="#4c78a8"), showlegend=False), row=1, col=1)
lim = [0, float(max(Etrue.max(), Epred.max()))]
fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line=dict(color="crimson", dash="dash"), showlegend=False), row=1, col=1)

bins = np.linspace(1, 100, 8); ctr = (bins[:-1] + bins[1:]) / 2
sig, bias = [], []
for i in range(len(ctr)):
    msk = (Etrue >= bins[i]) & (Etrue < bins[i + 1])
    if msk.sum() > 15:
        rb = resolution(Epred[msk], Etrue[msk]); sig.append(rb["sigma_eff"]); bias.append(rb["bias"])
    else:
        sig.append(None); bias.append(None)
fig.add_trace(go.Scatter(x=ctr, y=sig, mode="lines+markers", name="sigma_eff", line=dict(color="#4c78a8")), row=1, col=2)
fig.add_trace(go.Scatter(x=ctr, y=bias, mode="lines+markers", name="bias", line=dict(color="#e45756")), row=1, col=2)
fig.update_xaxes(title_text="true energy (GeV)")
fig.update_yaxes(title_text="predicted (GeV)", row=1, col=1)
fig.update_yaxes(title_text="resolution / bias", row=1, col=2)
fig.update_layout(template="plotly_white", height=430, title="B. Where does the kNN-25 model fail?")
fig.show()

In [21]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def err_analysis(K):
    rr = res[K]; model = rr["model"].eval()
    toks = rr["toks"]; Et = rr["Et"]; yv = rr["y"]; rte = rr["rte"]; rva = rr["rva"]
    mean, std = model._mean, model._std
    def predict(idxs):
        tens = []
        for i in idxs:
            t = toks[i].copy(); t[:, :7] = (t[:, :7] - mean) / std
            tens.append((torch.tensor(t), torch.tensor([0.0])))
        X, m, _ = collate(tens)
        with torch.no_grad():
            return model(X.to(DEVICE), m.to(DEVICE)).cpu().numpy().ravel()
    pv = predict(rva); a, b = np.polyfit(pv, yv[rva], 1)
    pt = predict(rte); Epred = np.exp(a * pt + b); Etrue = Et[rte]
    cont = np.array([np.expm1(toks[i][:, 0]).sum() / 1000.0 / max(Et[i], 1e-6) for i in rte])
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=[f"kNN-{K}: pred vs true (color=containment)", "resolution vs true energy"])
    fig.add_trace(go.Scatter(x=Etrue, y=Epred, mode="markers",
                             marker=dict(size=5, color=np.clip(cont, 0, 1.5), colorscale="RdYlGn",
                                         showscale=True, colorbar=dict(title="contain")), showlegend=False), row=1, col=1)
    lim = [0, float(max(Etrue.max(), Epred.max()))]
    fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line=dict(color="black", dash="dash"), showlegend=False), row=1, col=1)
    bins = np.linspace(1, 100, 8); ctr = (bins[:-1] + bins[1:]) / 2; sig = []
    for i in range(len(ctr)):
        msk = (Etrue >= bins[i]) & (Etrue < bins[i + 1])
        sig.append(resolution(Epred[msk], Etrue[msk])["sigma_eff"] if msk.sum() > 15 else None)
    fig.add_trace(go.Scatter(x=ctr, y=sig, mode="lines+markers", name=f"kNN-{K}"), row=1, col=2)
    fig.update_layout(template="plotly_white", height=420, title=f"Error analysis kNN-{K}")
    fig.update_xaxes(title_text="true energy (GeV)"); fig.update_yaxes(title_text="predicted (GeV)", row=1, col=1)
    fig.show()

err_analysis(9)
err_analysis(25)


In [6]:
!pip install run_experiments

In [8]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import build, select_knn, resolution, collate, TokenDS, EPS

class TunedTransformer(nn.Module):
    def __init__(self, in_dim, d=96, nhead=4, layers=3, dropout=0.1):
        super().__init__()
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, nhead, dim_feedforward=4 * d, dropout=dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d, 1))
    def forward(self, x, m):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = m.unsqueeze(-1).float()
        return self.head((h * w).sum(1) / w.sum(1).clamp(min=1))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
files = sorted((repo / "data" / "full").glob("matched_*.root"))[:100]

res = {}
for K in [9, 25]:
    ckpt = torch.load(repo / "models" / f"nb08_kNN{K}.pt", weights_only=False)
    model = TunedTransformer(ckpt["in_dim"], **ckpt["cfg"]).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"]); model.eval()
    model._mean, model._std = ckpt["mean"], ckpt["std"]
    D = build(files, 3, 100.0, selector=lambda c, k=K: select_knn(c, k))
    res[K] = {"model": model, "toks": D["tok_seed"], "Et": D["Etrue"], "y": D["y"],
              "rte": ckpt["rte"], "rva": ckpt["rva"]}
    print(f"loaded kNN-{K}: {len(res[K]['rte'])} test clusters")
print("DEVICE:", DEVICE)


loaded kNN-9: 1686 test clusters
loaded kNN-25: 1706 test clusters
DEVICE: cuda


In [9]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def error_correlate(K):
    rr = res[K]; model = rr["model"].eval()
    toks = rr["toks"]; Et = rr["Et"]; yv = rr["y"]; rte = rr["rte"]; rva = rr["rva"]
    mean, std = model._mean, model._std
    def predict(idxs):
        tens = []
        for i in idxs:
            t = toks[i].copy(); t[:, :7] = (t[:, :7] - mean) / std
            tens.append((torch.tensor(t), torch.tensor([0.0])))
        X, m, _ = collate(tens)
        with torch.no_grad():
            return model(X.to(DEVICE), m.to(DEVICE)).cpu().numpy().ravel()
    pv = predict(rva); a, b = np.polyfit(pv, yv[rva], 1)
    pt = predict(rte); Epred = np.exp(a * pt + b); Etrue = Et[rte]
    r = (Epred - Etrue) / Etrue
    cont = np.array([np.expm1(toks[i][:, 0]).sum() / 1000.0 / max(Et[i], 1e-6) for i in rte])
    fig = make_subplots(rows=1, cols=2, subplot_titles=[f"kNN-{K}: residual vs containment", "residual vs true energy"])
    fig.add_trace(go.Scatter(x=cont, y=r, mode="markers",
                             marker=dict(size=5, color=Etrue, colorscale="Viridis", showscale=True,
                                         colorbar=dict(title="E_true", x=0.46)), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=Etrue, y=r, mode="markers",
                             marker=dict(size=5, color=np.clip(cont, 0, 1.5), colorscale="RdYlGn",
                                         showscale=True, colorbar=dict(title="C")), showlegend=False), row=1, col=2)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)
    fig.update_xaxes(title_text="containment C", row=1, col=1); fig.update_xaxes(title_text="true energy (GeV)", row=1, col=2)
    fig.update_yaxes(title_text="residual (pred-true)/true")
    fig.update_layout(template="plotly_white", height=430, title=f"kNN-{K}: what explains the errors?")
    fig.show()

error_correlate(25)
error_correlate(9)

In [24]:
import torch
from pathlib import Path

mdl_dir = repo / "models"; mdl_dir.mkdir(exist_ok=True)
for K in [9, 25]:
    m = res[K]["model"]
    torch.save({"state_dict": m.state_dict(), "mean": m._mean, "std": m._std,
                "in_dim": m.embed.in_features,
                "cfg": {"d": cfg["d"], "nhead": cfg["nhead"], "layers": cfg["layers"], "dropout": cfg["dropout"]},
                "k": K, "rte": res[K]["rte"], "rva": res[K]["rva"]},
               mdl_dir / f"nb08_kNN{K}.pt")
    print("saved", mdl_dir / f"nb08_kNN{K}.pt")



K = 25
ckpt = torch.load(repo / "models" / f"nb08_kNN{K}.pt", weights_only=False)
model = TunedTransformer(ckpt["in_dim"], **ckpt["cfg"]).to(DEVICE)
model.load_state_dict(ckpt["state_dict"]); model.eval()
model._mean, model._std = ckpt["mean"], ckpt["std"]
D, keep = build_sel(K)
res = {K: {"model": model, "toks": D["tok_seed"], "Et": D["Etrue"],
           "y": D["y"], "rte": ckpt["rte"], "rva": ckpt["rva"]}}
# now err_analysis(K) and show_failures(K) work

saved ./models/nb08_kNN9.pt
saved ./models/nb08_kNN25.pt


If attention concentrates near the seed (distance ~0) and on the high-energy cells, the transformer is focusing on the physical shower core - the weight-vs-distance view Carla asked for, plus per-cluster event displays with attention overlaid on the true test data.